# 05B AI code test bench

**Problem:** Your team receives different code suggestions for an expense-splitting function. Build a fair comparison using the same inputs and expected results, review a candidate and decide what evidence supports your choice.

Inspired by [Code Model Comparison](https://github.com/patchy631/ai-engineering-hub/tree/042aeb5067384981eeba6a07fa882f436bb1660d/code-model-comparison). The original app compares hosted models with LiteLLM, Opik and a Streamlit interface. This independent classroom adaptation uses standard Python and direct behavioural checks.

**40-minute core:** 5 minutes understand the contract; 10 run the scorecard; 10 obtain/review a candidate; 10 add a case and feature; 5 explain.

Provided functions are labelled SAMPLE_IMPLEMENTATION_A and SAMPLE_IMPLEMENTATION_B. They are workshop examples, not outputs from named AI models. No API call is made by this notebook.


## 1 Define the task and expected example

**What this cell does:** Loads standard modules and records the function contract.

**Explain:** Inputs and expected behaviour must be fixed before comparing candidates. Money is represented as integer paise.

**Expected:** The example totals 30,000 paise, with a 10,000-paise share and balances +2,000, +8,000 and -10,000.


In [ ]:
import copy, json
MODE = "LOCAL_BEHAVIOUR_TESTS"
PARTICIPANTS = ["Asha", "Bala", "Charan"]
EXPENSES = [{"payer":"Asha", "amount_paise":12000}, {"payer":"Bala", "amount_paise":18000}]
EXPECTED = {"total_paise":30000, "share_paise":10000,
            "balances_paise":{"Asha":2000, "Bala":8000, "Charan":-10000}}
CONTRACT = """split_expenses(participants, expenses) returns total_paise, share_paise and balances_paise.
Participants must be a nonempty list of unique nonempty strings.
Each expense has exactly payer and amount_paise; the payer must be a participant.
Each amount is a positive integer; booleans are invalid.
For this core exercise, the total must divide evenly among participants; otherwise raise ValueError.
Empty expense lists are allowed and give zero balances.
Positive balance means the person should receive money; negative means they should contribute.
Invalid input raises ValueError. Do not mutate the supplied inputs."""
print(CONTRACT)
print("Example expected result:", json.dumps(EXPECTED, indent=2))


## 2 Inspect the labelled sample implementations

**What this cell does:** Defines two small, supplied Python functions. One is plausible but violates several requirements; one handles the stated contract.

**Explain:** Read the code before testing it. Neither label represents a real model's performance. These checks apply only to this specified exercise.

**Expected:** Definitions only. Test results appear in the next step.


In [ ]:
def SAMPLE_IMPLEMENTATION_A(participants, expenses):
    # Supplied classroom sample: plausible arithmetic is not enough.
    paid = {name:0 for name in participants}
    for expense in expenses:
        paid[expense["payer"]] += int(expense["amount_paise"])
    total = sum(paid.values())
    share = total // len(participants)
    return {"total_paise":total, "share_paise":share,
            "balances_paise":{name:share-paid[name] for name in participants}}

def SAMPLE_IMPLEMENTATION_B(participants, expenses):
    # Validate the argument contract before calculating or changing anything.
    if not isinstance(participants, list) or not participants or any(not isinstance(name,str) or not name.strip() for name in participants):
        raise ValueError("Use a nonempty list of participant names")
    if len(set(participants)) != len(participants):
        raise ValueError("Participants must be unique")
    if not isinstance(expenses, list):
        raise ValueError("Expenses must be a list")
    paid = {name:0 for name in participants}
    for expense in expenses:
        if not isinstance(expense,dict) or set(expense) != {"payer","amount_paise"}:
            raise ValueError("Use exactly payer and amount_paise")
        payer, amount = expense["payer"], expense["amount_paise"]
        if not isinstance(payer,str) or payer not in paid:
            raise ValueError("Unknown payer")
        if type(amount) is not int or amount <= 0:
            raise ValueError("Amount must be a positive integer number of paise")
        paid[payer] += amount
    total = sum(paid.values())
    if total % len(participants):
        raise ValueError("Core exercise requires an evenly divisible total")
    share = total // len(participants)
    return {"total_paise":total, "share_paise":share,
            "balances_paise":{name:paid[name]-share for name in participants}}


## 3 Keep one common test contract

**What this cell does:** Defines ten checks used unchanged for every candidate.

**Explain:** A comparison is unfair if each candidate gets different expected values. The runner catches individual failures and reports them as results; a failed sample is not an unavailable notebook runtime.

**Expected:** Ten tests are ready. No model, installation or network operation occurs.


In [ ]:
def require(condition, message="Result differs from the contract"):
    if not condition:
        raise AssertionError(message)

def expect_value_error(operation):
    try:
        operation()
    except ValueError:
        return
    raise AssertionError("Expected ValueError")

def test_example(candidate):
    require(candidate(copy.deepcopy(PARTICIPANTS),copy.deepcopy(EXPENSES)) == EXPECTED)
def test_equal_payments(candidate):
    result = candidate(["A","B"],[{"payer":"A","amount_paise":100},{"payer":"B","amount_paise":100}])
    require(result == {"total_paise":200,"share_paise":100,"balances_paise":{"A":0,"B":0}})
def test_empty_expenses(candidate):
    require(candidate(["A","B"],[]) == {"total_paise":0,"share_paise":0,"balances_paise":{"A":0,"B":0}})
def test_single_payer(candidate):
    require(candidate(["A","B"],[{"payer":"A","amount_paise":200}]) ==
            {"total_paise":200,"share_paise":100,"balances_paise":{"A":100,"B":-100}})
def test_unknown_payer(candidate):
    expect_value_error(lambda:candidate(["A","B"],[{"payer":"X","amount_paise":200}]))
def test_boolean_amount(candidate):
    expect_value_error(lambda:candidate(["A","B"],[{"payer":"A","amount_paise":True}]))
def test_negative_amount(candidate):
    expect_value_error(lambda:candidate(["A","B"],[{"payer":"A","amount_paise":-200}]))
def test_duplicate_names(candidate):
    expect_value_error(lambda:candidate(["A","A"],[]))
def test_indivisible_total(candidate):
    expect_value_error(lambda:candidate(["A","B"],[{"payer":"A","amount_paise":101}]))
def test_inputs_unchanged(candidate):
    names, expenses = copy.deepcopy(PARTICIPANTS), copy.deepcopy(EXPENSES)
    old_names, old_expenses = copy.deepcopy(names), copy.deepcopy(expenses)
    candidate(names,expenses)
    require(names == old_names and expenses == old_expenses)

FIXED_CHECKS = [test_example,test_equal_payments,test_empty_expenses,test_single_payer,test_unknown_payer,
                test_boolean_amount,test_negative_amount,test_duplicate_names,test_indivisible_total,test_inputs_unchanged]
def evaluate_candidate(label, candidate, checks=FIXED_CHECKS):
    outcomes = []
    for check in checks:
        try:
            check(candidate)
            outcomes.append({"check":check.__name__,"status":"PASS"})
        except Exception as error:
            # Report the failed requirement without an unhandled red traceback.
            outcomes.append({"check":check.__name__,"status":"FAIL","issue":type(error).__name__})
    return {"label":label,"passed":sum(row["status"]=="PASS" for row in outcomes),
            "total":len(outcomes),"outcomes":outcomes}
print("Fixed common checks:", len(FIXED_CHECKS))


## 4 Run the scorecard and inspect the differences

**What this cell does:** Runs the same ten checks on both samples.

**Student task 1:** Choose two failed requirements in sample A and explain which code causes the mismatch. Do not change the tests to give it a better score.

**Expected:** SAMPLE_IMPLEMENTATION_A passes 3/10 and SAMPLE_IMPLEMENTATION_B passes 10/10. These are observed exercise results, not a general ranking of AI models.


In [ ]:
sample_reports = [evaluate_candidate("SAMPLE_IMPLEMENTATION_A",SAMPLE_IMPLEMENTATION_A),
                  evaluate_candidate("SAMPLE_IMPLEMENTATION_B",SAMPLE_IMPLEMENTATION_B)]
for report in sample_reports:
    print(f"{report['label']}: {report['passed']} / {report['total']} passed")
    for outcome in report["outcomes"]:
        print(" ", outcome["check"], outcome["status"], outcome.get("issue",""))
print("Sample reports are supplied-code checks; no model comparison is claimed.")


## 5 Obtain a candidate using a precise request

**What this cell does:** Prints the same contract and expected example for your chosen assistant.

**Student task 2:** Obtain a candidate from an available assistant, or write one yourself. Record the actual route, model if known and prompt. You may compare two prompts with one model; two paid accounts are not required.

**Expected:** A prompt to copy. This cell does not generate or execute an external answer.


In [ ]:
CANDIDATE_PROMPT = f"""Implement one pure Python function split_expenses(participants, expenses).
{CONTRACT}
Example participants: {PARTICIPANTS}
Example expenses: {EXPENSES}
Expected example: {EXPECTED}
Add no dependencies, filesystem access, shell commands or network calls.
Explain your validation and provide the function. Do not claim tests were run.
"""
print(CANDIDATE_PROMPT)


## 6 Review your candidate before calling it

**What this cell does:** Provides a place for a student-reviewed function and provenance. It does not automatically load generated text, execute a model response or install a package.

Replace only your candidate function after reading it. Set REVIEWED_FOR_EXECUTION when you understand it and are ready to run it against the fixed checks. Python cells execute normally; the flag is a workflow reminder, not a sandbox.

**Expected:** NOT_RUN until a real candidate is supplied and explicitly reviewed.


In [ ]:
# Replace None with your reviewed function object after defining it in this cell.
STUDENT_CANDIDATE = None
REVIEWED_FOR_EXECUTION = False
PROVENANCE = {"route":"not recorded", "model_if_known":"", "prompt_used":"", "changes_after_review":""}
student_report = {"label":"STUDENT_CANDIDATE","status":"NOT_RUN"}
if REVIEWED_FOR_EXECUTION and callable(STUDENT_CANDIDATE):
    student_report = evaluate_candidate("STUDENT_CANDIDATE",STUDENT_CANDIDATE)
print(json.dumps(student_report,indent=2))


## 7 Add an edge case and one useful feature

**Student task 3:** Add your own check, such as a fractional amount, empty participant list or an extra expense field. Apply it to every candidate fairly. Then add a small feature such as a readable balance summary; keep the money calculation unchanged.

**What this cell does:** Runs only the additional functions you choose to supply. Empty lists mean no student extension has been checked.

**Expected:** A truthful NOT_RUN message until you implement the extra check or feature. A passing original sample does not complete this task for you.


In [ ]:
STUDENT_CHECKS = []  # Add functions taking one candidate, with independent expected values.
BALANCE_SUMMARY = None  # Optional reviewed function accepting the result dictionary and returning text.
extra_reports = []
if STUDENT_CHECKS:
    for name, candidate in [("SAMPLE_IMPLEMENTATION_A",SAMPLE_IMPLEMENTATION_A),("SAMPLE_IMPLEMENTATION_B",SAMPLE_IMPLEMENTATION_B)]:
        extra_reports.append(evaluate_candidate(name,candidate,STUDENT_CHECKS))
    if REVIEWED_FOR_EXECUTION and callable(STUDENT_CANDIDATE):
        extra_reports.append(evaluate_candidate("STUDENT_CANDIDATE",STUDENT_CANDIDATE,STUDENT_CHECKS))
print("Additional checks:", json.dumps(extra_reports,indent=2) if extra_reports else "NOT_RUN")
feature_result = {"status":"NOT_RUN"}
if callable(BALANCE_SUMMARY):
    # The summary should read the completed result without changing its numbers.
    before_result = copy.deepcopy(EXPECTED)
    try:
        summary = BALANCE_SUMMARY(before_result)
        feature_result = {"status":"OBSERVED", "summary":summary, "input_unchanged":before_result==EXPECTED}
    except Exception as error:
        feature_result = {"status":"FAIL","issue":type(error).__name__}
print("Feature observation:",feature_result)


## 8 Save a fair conclusion

**What this cell does:** Records the common scorecard and whether you actually evaluated a reviewed student candidate and extra checks.

Explain what the tests establish and what they do not. More passing examples do not prove a function is correct for every possible input, or that a named model is always better.


In [ ]:
MY_CONCLUSION = ""  # Explain one requirement, observed result and remaining limitation.
evidence = {"mode":MODE,"sample_reports":sample_reports,"student_report":student_report,
            "provenance":PROVENANCE,"extra_reports":extra_reports,"feature":feature_result,
            "my_conclusion":MY_CONCLUSION,"model_requests_by_this_notebook":0}
print(json.dumps(evidence,indent=2))


## Show your neighbour

1. Explain one result using the original contract and a specific input.
2. Show your reviewed candidate and actual provenance, or label your work as a manual implementation.
3. Show an additional check and your feature observation without changing the old expected values.

**Credit:** Project inspiration: patchy631, AI Engineering Hub, Code Model Comparison, pinned commit 042aeb5067384981eeba6a07fa882f436bb1660d. This is original classroom code using direct behavioural checks; it does not reproduce the upstream app or claim to run its hosted-model or Opik evaluation.
